In [2]:
# import libraries

import pandas as pd
import xarray as xr
import numpy as np
from pathlib import Path
import seaborn as sns
from matplotlib import pyplot as plt
from scipy.stats import kruskal

import warnings
warnings.filterwarnings('ignore')

In [3]:
data = pd.read_csv('../../local_data/validation_all.csv')

In [4]:
random_sample_images = data.sample(n=40, random_state=42)
random_sample_images.to_csv('../../local_data/random_sample_images.csv', index=False)

In [1]:
import pandas as pd

# ---------------------------------------------------------
# ASSUMPTION: 'data' is already loaded in your environment 
# and contains the combined pipeline results across all years.
# ---------------------------------------------------------

print("Calculating errors and finding extreme locations across all years...")

# 1. Ensure the absolute error column exists
# (If you already calculated this in your notebook, this will just overwrite/update it safely)
if 'team_error' not in data.columns:
    data['team_error'] = data['team_icecon'] - data['visual_ice']
    
data['team_abs_error'] = data['team_error'].abs()

# 2. Group by row and col to get the mean error and observation count (n) across ALL time periods
cell_stats = data.groupby(['row', 'col']).agg(
    mean_abs_error=('team_abs_error', 'mean'),
    n=('team_abs_error', 'count')
).reset_index()

# 3. Filter for cells with more than 10 observations
valid_cells = cell_stats[cell_stats['n'] > 10]

# 4. Find the 5 areas with the highest and lowest mean absolute error
highest_error_cells = valid_cells.nlargest(5, 'mean_abs_error')
lowest_error_cells = valid_cells.nsmallest(5, 'mean_abs_error')

# Print them out for your records
print("\n--- Top 5 Areas with HIGHEST Error (n > 10) ---")
print(highest_error_cells.to_string(index=False))

print("\n--- Top 5 Areas with LOWEST Error (n > 10) ---")
print(lowest_error_cells.to_string(index=False))

# 5. Combine the 10 target cells into one dataframe
target_cells = pd.concat([highest_error_cells, lowest_error_cells])

# 6. Filter the original multi-year dataframe to keep ONLY the data from these 10 areas
# Using an inner merge gracefully drops any rows from 'data' that aren't in 'target_cells'
subset_data = data.merge(target_cells[['row', 'col']], on=['row', 'col'], how='inner')

# 7. Save the subset out to a new CSV
# Note: Removed the {year} string formatting since this represents the combined dataset
subset_outpath = '../../local_data/DataFrames/extreme_errors_subset_all_years.csv'
subset_data.to_csv(subset_outpath, index=False)

print(f"\nSaved subset of {len(subset_data)} rows for the 10 target areas to: {subset_outpath}")

Calculating errors and finding extreme locations across all years...


NameError: name 'data' is not defined